#### Institute of Data Capstone Project
# 📚 Books Don't Die, They Slow Down
##### Notebook 1 of 4 · Data Contract
##### Author: Mike Ee, [mikeee.co](https://mikeee.co)

Research Question
> Which lines in a 15-year invoice register count as book sales, and how is every other line accounted for?

## Table of Contents

1. [Background](#background)
2. [Setup](#setup)
3. [Load and prove provenance](#load)
4. [Book scope from `Type`](#scope)
5. [Line taxonomy](#taxonomy)
6. [Title master](#master)
7. [Genre and sub-category](#genre)
8. [Catalogue gaps](#gaps)
9. [Filter contract](#contract)
10. [Export](#export)
11. [Conclusion](#conclusion)

<a id="background"></a>
## 1. Background

The client's invoice register holds 94,872 lines over 180 months (August 2011 to July 2026). An earlier cleaning pass produced `sales_clean.parquet`: ISBN normalisation, date reconstruction, genre standardisation and Culture Pass flags. The register mixes book sales with services, shipping, vouchers and invoice furniture, so a book line must be **defined positively** before any analysis.

This notebook builds that definition once and serialises it as a **filter contract**. Every later notebook reads the contract through `capstone.py` instead of retyping filter strings.

**Ordering principle.** `Type`, the invoiced format, **selects** book lines. `Amount`, `FOC` and `Qty` then **classify** them. Filtering on `Amount > 0` before building the title list would silently delete every free-of-charge (FOC) and unpriced title.

| Output | Contents |
|---|---|
| `data/processed/sales_scoped.parquet` | all 94,872 lines, plus `book_scope`, `scope_reason`, `line_role`, sign and genre flags |
| `data/lookup/book_master.csv` | one row per `isbn_key` |
| `data/lookup/analysis_filters.json` | the filter contract |

<a id="setup"></a>
## 2. Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from capstone import pseudonym, mask_index, mask_cols   # display aliases for titles and ISBNs

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

PROJECT   = Path(".")
DATA      = PROJECT / "data"
PROCESSED = DATA / "processed"
LOOKUP    = DATA / "lookup"

CONTRACT_VERSION = 1          # bump on ANY change to scope or a filter string
FROZEN_ROWS      = 94_872     # register size; every row-count assert keys off this

# engine="python" on every .query(): numexpr cannot evaluate pandas extension
# dtypes (client_excluded is nullable `boolean`).
Q = dict(engine="python")

print(f"pandas {pd.__version__} · contract v{CONTRACT_VERSION}")

pandas 2.3.3 · contract v1


<a id="load"></a>
## 3. Load and prove provenance

The contract rests on one claim: **`sales_clean.parquet` is the raw register plus derived columns, with no rows removed.** Section 3.2 tests that claim against the raw register and is designed to fail loudly.

In [2]:
# ── 3.1 load ─────────────────────────────────────────────────────────
src = pd.read_parquet(PROCESSED / "sales_clean.parquet")
print(f"sales_clean : {len(src):,} rows x {src.shape[1]} cols")
print(f"date span   : {src['date'].min():%Y-%m-%d} -> {src['date'].max():%Y-%m-%d}")
print(f"months      : {src['ym'].nunique()}")

sales_clean : 94,872 rows x 47 cols
date span   : 2011-08-01 -> 2026-07-31
months      : 180


In [3]:
# ── 3.2 provenance gate: must pass before anything downstream ───────
# Objective: prove sales_clean is a faithful superset of the raw register.
# MODIFY CONDITION: if the raw file is unavailable, the gate falls back to
#   the row-count check and says so. Do not delete the check.
RAW_PATH       = DATA / "raw" / "invoice_register.parquet"
SKIP_RAW_CHECK = not RAW_PATH.exists()

assert len(src) == FROZEN_ROWS, (
    f"sales_clean is {len(src):,} rows, expected {FROZEN_ROWS:,}. "
    "A row-removing step entered the cleaning pass: stop and locate it.")

if SKIP_RAW_CHECK:
    print(f"raw register not found at {RAW_PATH}: row-count gate only")
else:
    raw = pd.read_parquet(RAW_PATH)
    print(f"raw register: {len(raw):,} rows x {raw.shape[1]} cols")

    assert len(raw) == len(src), (
        f"row count differs: raw {len(raw):,} vs sales_clean {len(src):,}. "
        "The 'no rows removed' rule is violated: stop.")

    # Money is the strongest fingerprint: it survives every rename and recode.
    _raw_amt = pd.to_numeric(raw["Amount"], errors="coerce").sum()
    _src_amt = src["Amount"].sum()
    print(f"Amount  raw vs clean delta: {abs(_raw_amt - _src_amt):,.2f}")
    assert abs(_raw_amt - _src_amt) < 1.0, "Amount total drifted between raw and clean"

    _raw_qty = pd.to_numeric(raw["Qty"], errors="coerce").sum()
    print(f"Qty     raw {_raw_qty:,.0f} vs clean {src['Qty'].sum():,.0f}")
    assert abs(_raw_qty - src["Qty"].sum()) < 1.0, "Qty total drifted"

print("\n[OK] provenance gate passed: sales_clean is register-faithful")

raw register: 94,872 rows x 19 cols
Amount  raw vs clean delta: 0.00
Qty     raw 653,585 vs clean 653,585

[OK] provenance gate passed: sales_clean is register-faithful


In [4]:
# ── 3.3 columns the contract depends on ─────────────────────────────
NEEDED = ["Type", "Qty", "Amount", "FOC", "Item ID", "Item Description",
          "isbn_key", "date", "ym", "Invoice/CM #", "Customer ID"]
missing = [c for c in NEEDED if c not in src.columns]
assert not missing, f"sales_clean lacks required columns: {missing}"
print(f"all required columns present ({len(NEEDED)})")

all required columns present (11)


<a id="scope"></a>
## 4. Book scope from `Type`

`Type` is the format as invoiced: the register's own statement about what the line is, and authoritative over `edition`. Scope is a **positive** test. The residual is named and audited, never silently absorbed.

In [5]:
# ── 4.1 what is actually in Type ─────────────────────────────────────
print("Type inventory, unfiltered register")
print(src["Type"].value_counts(dropna=False).to_string())
print(f"\nnulls: {src['Type'].isna().sum():,}")

# Whitespace/case duplicates would split one format into two categories.
_raw_vals = src["Type"].dropna().unique()
_norm = pd.Series(_raw_vals).str.strip().str.lower()
_dupes = _norm[_norm.duplicated(keep=False)]
print(f"\nwhitespace/case duplicate Type values: {len(_dupes)}")
assert len(_dupes) == 0, f"Type needs normalisation first: {sorted(set(_dupes))}"

Type inventory, unfiltered register
Type
Paperback                  86513
eBook                       4840
Shipping/Postal Charges     1808
Not recorded                1288
Others                       254
Gift Voucher                 169



nulls: 0

whitespace/case duplicate Type values: 0


In [6]:
# ── 4.2 scope sets ───────────────────────────────────────────────────
# MODIFY CONDITION: add a Type to BOOK_TYPES only after §4.4 shows it carries
#   isbn_key values. Adding on intuition turns the definition into a residual.
BOOK_TYPES = {"Paperback", "eBook"}

# Carve-outs that must survive Type selection, established on Item ID evidence
# during cleaning: an `lp` line is a service fee regardless of how Type was coded.
SERVICE_IDS   = {"lp"}
INSTITUTIONAL = "MOE"

_present = set(src["Type"].dropna().unique())
assert BOOK_TYPES <= _present, f"BOOK_TYPES not in data: {BOOK_TYPES - _present}"

m_service = src["Item ID"].isin(SERVICE_IDS)
m_instit  = src["Item ID"].astype(str).str.startswith(INSTITUTIONAL)
m_type    = src["Type"].isin(BOOK_TYPES)

# Precedence: service > institutional > book.
# A book Type does NOT rescue a line that is a service fee.
src["book_scope"] = m_type & ~m_service & ~m_instit

print(f"book_scope True  : {src['book_scope'].sum():,}")
print(f"book_scope False : {(~src['book_scope']).sum():,}")
print(f"\n  service carve-out    : {m_service.sum():,}")
print(f"  institutional carve  : {m_instit.sum():,}")
print(f"  non-book Type        : {(~m_type).sum():,}")
print(f"  book Type overridden : {(m_type & (m_service | m_instit)).sum():,}")
# ^ the last line is the value of keeping precedence. If it is >0, a pure
#   Type filter would have folded service/print-run lines into book demand.

book_scope True  : 91,257
book_scope False : 3,615

  service carve-out    : 175
  institutional carve  : 96
  non-book Type        : 3,519
  book Type overridden : 96


In [7]:
# ── 4.3 scope_reason: every excluded line gets a stated cause ───────
src["scope_reason"] = np.select(
    [src["book_scope"], m_service, m_instit],
    ["book", "service", "institutional_bulk"],
    default="non_book_type")

print(pd.crosstab(src["scope_reason"], src["Type"], dropna=False).to_string())

summary = (src.groupby("scope_reason")
             .agg(lines=("Qty", "size"),
                  units=("Qty", "sum"),
                  revenue=("Amount", "sum"),
                  isbns=("isbn_key", "nunique")))
summary["revenue_%"] = (summary["revenue"] / summary["revenue"].sum() * 100).round(2)
print(f"\n{summary.drop(columns='revenue').to_string()}")

assert summary["lines"].sum() == len(src), "scope_reason does not partition the register"

Type                Gift Voucher  Not recorded  Others  Paperback  Shipping/Postal Charges  eBook
scope_reason                                                                                     
book                           0             0       0      86417                        0   4840
institutional_bulk             0             0       0         96                        0      0
non_book_type                169          1113     254          0                     1808      0
service                        0           175       0          0                        0      0

                    lines      units  isbns  revenue_%
scope_reason                                          
book                91257  497958.00    640      76.53
institutional_bulk     96   13273.00      0       2.44
non_book_type        3344  142167.51      1      10.02
service               175     186.00      0      11.00


In [8]:
# ── 4.4 residual audit: the cell that justifies BOOK_TYPES ─────────
# An excluded line carrying an isbn_key is a contradiction: either the Type is
# wrong, or the key was wrongly inherited. Name every one.
resid = src[src["scope_reason"] == "non_book_type"]
print(f"non-book-Type lines: {len(resid):,} | {resid['Qty'].sum():,.0f} units "
      f"| {resid['Amount'].sum() / src['Amount'].sum() * 100:.1f}% of register revenue")
print(f"\n{resid['Type'].value_counts(dropna=False).to_string()}")

orphan = resid[resid["isbn_key"].notna()]
print(f"\n>>> excluded but ISBN-keyed: {len(orphan):,} lines, "
      f"{orphan['isbn_key'].nunique():,} distinct ISBNs <<<")
if len(orphan):
    print(orphan.groupby("Type").agg(lines=("Qty", "size"),
                                     isbns=("isbn_key", "nunique"),
                                     units=("Qty", "sum")).to_string())
    print("\ntop Item Descriptions among orphans (aliased):")
    print(mask_index(orphan["Item Description"].value_counts().head(15), "Item").to_string())

# Deliberately NOT an assert. Orphans may be legitimate (a voucher redeemed
# against a title). This print is the evidence for the rescue decision in 4.5.

non-book-Type lines: 3,344 | 142,168 units | 10.0% of register revenue

Type
Shipping/Postal Charges    1808
Not recorded               1113
Others                      254
Gift Voucher                169

>>> excluded but ISBN-keyed: 1 lines, 1 distinct ISBNs <<<
              lines  isbns  units
Type                             
Not recorded      1      1   27.0

top Item Descriptions among orphans (aliased):
Item 001    1


In [9]:
# ── 4.5 optional rescue ──────────────────────────────────────────────
# A switch with the consequence measured either way.
# MODIFY CONDITION: set False to keep ISBN-keyed non-book lines out of scope.
RESCUE_ISBN_ORPHANS = True

if RESCUE_ISBN_ORPHANS:
    rescue = (src["scope_reason"] == "non_book_type") & src["isbn_key"].notna()
    src.loc[rescue, "book_scope"]   = True
    src.loc[rescue, "scope_reason"] = "book_rescued_isbn"
    print(f"RESCUED {rescue.sum():,} lines on isbn_key evidence")
else:
    _would = ((src["scope_reason"] == "non_book_type") & src["isbn_key"].notna()).sum()
    print(f"rescue OFF: {_would:,} ISBN-keyed lines remain out of scope")

print(f"\nfinal book_scope: {src['book_scope'].sum():,} lines "
      f"({src['book_scope'].mean()*100:.2f}% of register)")

RESCUED 1 lines on isbn_key evidence

final book_scope: 91,258 lines (96.19% of register)


<a id="taxonomy"></a>
## 5. Line taxonomy: `Amount`, `Qty`, `FOC`

Each numeric column gets **four** states, not three. `NaN == 0` evaluates `False` in pandas, so a negative/zero/positive split silently drops every null line into no bucket. `Amount` has about 10,000 nulls; a three-way split would lose over 10% of the register without a warning.

Sign-of-money and sign-of-units are **orthogonal**. Some lines carry `Amount < 0` with positive `Qty`: discounts, not returns. Collapsing the two axes would break the gross-to-net bridge in notebook 2.

In [10]:
# ── 5.1 sign classifier ──────────────────────────────────────────────
def sign_class(s: pd.Series) -> pd.Series:
    """Four-state sign classification. Null is a state, not an absence."""
    return pd.Series(
        np.select([s.isna(), s < 0, s == 0, s > 0],
                  ["null", "negative", "zero", "positive"],
                  default="unclassified"),
        index=s.index, dtype="object")

src["amt_sign"] = sign_class(src["Amount"])
src["qty_sign"] = sign_class(src["Qty"])

assert (src["amt_sign"] != "unclassified").all(), "an Amount escaped classification"
assert (src["qty_sign"] != "unclassified").all(), "a Qty escaped classification"

print("Amount"); print(src["amt_sign"].value_counts().to_string())
print("\nQty");  print(src["qty_sign"].value_counts().to_string())

Amount


amt_sign
positive    78730
null         9989
negative     6153

Qty
qty_sign
positive    87099
negative     7411
null          362


In [11]:
# ── 5.2 FOC, from the client's native record ────────────────────────
# `FOC` is a presence marker (about 91% null, one distinct value), not a value
# column. Read notna(), never the string.
src["is_foc"] = src["FOC"].notna()

print(f"FOC-marked lines: {src['is_foc'].sum():,}")
print("\nFOC x Amount sign, the containment claim:")
print(pd.crosstab(src["is_foc"], src["amt_sign"], margins=True).to_string())

# Every FOC line should carry a null Amount, with zero exceptions.
_viol = (src["is_foc"] & src["Amount"].notna()).sum()
assert _viol == 0, (
    f"{_viol:,} FOC lines carry a price: the FOC/null-Amount containment "
    "relationship has broken and §5.3 needs revisiting")

# Null Amount NOT marked FOC = missing price (mostly delivery orders, doc_type 'DO').
src["price_missing"] = src["Amount"].isna() & ~src["is_foc"]
print(f"\nprice_missing (null Amount, not FOC): {src['price_missing'].sum():,}")

FOC-marked lines: 8,660

FOC x Amount sign, the containment claim:
amt_sign  negative  null  positive    All
is_foc                                   
False         6153  1329     78730  86212
True             0  8660         0   8660
All           6153  9989     78730  94872

price_missing (null Amount, not FOC): 1,329


In [12]:
# ── 5.3 line_role: one label per line, mutually exclusive ──────────
# Precedence is declared, not incidental. np.select returns on FIRST True.
#   structural  : no units recorded, invoice furniture (GST, subtotals)
#   return      : units came back (sign of Qty, never doc_type)
#   credit_adj  : money reversed, no units moved (discounts, rounding)
#   foc         : units moved, no money, deliberately
#   unpriced    : units moved, no money, NOT deliberate (data quality)
#   sale        : units and money both moved
cond = [
    src["qty_sign"] == "null",
    src["qty_sign"] == "negative",
    (src["amt_sign"] == "negative") & src["qty_sign"].isin(["zero", "positive"]),
    src["is_foc"],
    src["price_missing"],
]
src["line_role"] = np.select(
    cond, ["structural", "return", "credit_adj", "foc", "unpriced"],
    default="sale")

roles = (src.groupby("line_role")
           .agg(lines=("Qty", "size"), units=("Qty", "sum"),
                revenue=("Amount", "sum"), isbns=("isbn_key", "nunique")))
roles["revenue_%"] = (roles["revenue"] / roles.loc["sale", "revenue"] * 100).round(2)
print(roles.drop(columns="revenue").to_string())
print("  (revenue_% is relative to sale revenue)")
assert roles["lines"].sum() == len(src), "line_role does not partition the register"

print("\nline_role x book_scope")
print(pd.crosstab(src["line_role"], src["book_scope"], margins=True).to_string())

            lines      units  isbns  revenue_%
line_role                                     
credit_adj    179     229.00      0      -0.29
foc          7637   79342.00    485       0.00
return       7411  -47897.00    491      -6.19
sale        78399  605823.51    632     100.00
structural    362       0.00      0       0.86
unpriced      884   16087.00    214       0.00
  (revenue_% is relative to sale revenue)

line_role x book_scope


book_scope  False   True    All
line_role                      
credit_adj    179      0    179
foc           142   7495   7637
return         45   7366   7411
sale         2863  75536  78399
structural    362      0    362
unpriced       23    861    884
All          3614  91258  94872


In [13]:
# ── 5.4 the taxonomy crosstab ────────────────────────────────────────
bk = src[src["book_scope"]]
print(f"book-scoped lines: {len(bk):,}\n")
print(pd.crosstab(bk["qty_sign"], bk["amt_sign"], margins=True).to_string())
print("""
READ:
  qty positive / amt null     -> FOC + delivery orders (units, no money)
  qty positive / amt negative -> discounts, NOT returns
  qty negative / amt negative -> true returns
  qty negative / amt positive -> contradiction; investigate if material
  qty null                    -> invoice structure lines
""")

_contra = bk.query('qty_sign == "negative" and amt_sign == "positive"', **Q)
print(f"contradiction cell: {len(_contra):,} lines")
if len(_contra):
    print(_contra["doc_type"].value_counts().head().to_string()
          if "doc_type" in _contra else "")

book-scoped lines: 91,258



amt_sign  negative  null  positive    All
qty_sign                                 
negative      5918  1448         0   7366
positive         0  8356     75536  83892
All           5918  9804     75536  91258

READ:
  qty positive / amt null     -> FOC + delivery orders (units, no money)
  qty positive / amt negative -> discounts, NOT returns
  qty negative / amt negative -> true returns
  qty negative / amt positive -> contradiction; investigate if material
  qty null                    -> invoice structure lines

contradiction cell: 0 lines


<a id="master"></a>
## 6. Title master

Grain is **`isbn_key`**: one ISBN is one sellable edition. `title` is a label. A paperback and an eBook of the same work are two products with one title string, and collapsing them would destroy the format analysis.

Every in-scope line contributes, regardless of `line_role`. Activity flags record what a title actually did; nothing is filtered out at this step, which keeps FOC-only and unpriced-only titles in the catalogue.

In [14]:
# ── 6.1 aggregate to ISBN grain ──────────────────────────────────────
bk = src[src["book_scope"]].copy()

unkeyed = bk["isbn_key"].isna().sum()
print(f"book-scoped lines with no isbn_key: {unkeyed:,} "
      f"({unkeyed/len(bk)*100:.2f}%)")
if unkeyed:
    print(mask_index(bk.loc[bk['isbn_key'].isna(), 'Item Description']
                       .value_counts().head(10), "Item").to_string())
# ^ these cannot enter a title master. Listed in §8.

keyed = bk[bk["isbn_key"].notna()].copy()

def _sum_where(col, mask_role=None, sign=None):
    """Build a named aggregation over a role- or sign-restricted subset."""
    s = keyed[col]
    if mask_role is not None: s = s.where(keyed["line_role"].isin(mask_role))
    if sign == "pos":         s = s.where(keyed[col] > 0)
    if sign == "neg":         s = s.where(keyed[col] < 0)
    return s

keyed["_qty_pos"]  = _sum_where("Qty", sign="pos")
keyed["_qty_neg"]  = _sum_where("Qty", sign="neg")
keyed["_rev_sale"] = keyed["Amount"].where(keyed["line_role"] == "sale")

book_master = (keyed.groupby("isbn_key").agg(
        item_desc     = ("Item Description", "first"),
        n_desc        = ("Item Description", "nunique"),
        fmt_types     = ("Type", lambda s: "|".join(sorted(s.dropna().unique()))),
        lines         = ("Qty", "size"),
        units_gross   = ("_qty_pos", "sum"),
        units_returned= ("_qty_neg", "sum"),
        revenue_sale  = ("_rev_sale", "sum"),
        revenue_net   = ("Amount", "sum"),
        first_sale    = ("date", "min"),
        last_sale     = ("date", "max"),
        active_months = ("ym", "nunique"),
        n_customers   = ("Customer ID", "nunique"),
        n_invoices    = ("Invoice/CM #", "nunique"),
    ))

# role mix per ISBN, widened
role_mix = (keyed.pivot_table(index="isbn_key", columns="line_role",
                              values="Qty", aggfunc="size", fill_value=0)
              .add_prefix("n_"))
book_master = book_master.join(role_mix, how="left").fillna({c: 0 for c in role_mix})

print(f"\nbook_master: {len(book_master):,} unique isbn_key")
print(book_master[["lines", "units_gross", "active_months"]]
        .describe().round(2).to_string())

book-scoped lines with no isbn_key: 1,041 (1.14%)
Item 002    177
Item 003    103
Item 004     50
Item 005     46
Item 006     45
Item 007     33
Item 008     29
Item 009     29
Item 010     28
Item 011     27



book_master: 640 unique isbn_key
         lines  units_gross  active_months
count   640.00       640.00         640.00
mean    140.96       845.04          36.23
std     247.57      2306.67          31.68
min       1.00         0.00           1.00
25%      20.00        54.75          11.75
50%      62.00       320.50          27.00
75%     170.25       852.25          53.00
max    3666.00     38021.00         173.00


In [15]:
# ── 6.2 activity flags: classify, never filter ─────────────────────
# A title with no priced sale is still a title; it is flagged, not deleted.
book_master["has_priced_sale"] = book_master["revenue_sale"].fillna(0) > 0
book_master["has_any_units"]   = book_master["units_gross"].fillna(0) > 0
book_master["foc_only"]        = (~book_master["has_priced_sale"]
                                  & (book_master.get("n_foc", 0) > 0))
book_master["returns_only"]    = (~book_master["has_any_units"]
                                  & (book_master.get("n_return", 0) > 0))
book_master["multi_desc"]      = book_master["n_desc"] > 1

for c in ["has_priced_sale", "has_any_units", "foc_only", "returns_only", "multi_desc"]:
    print(f"{c:<18} {int(book_master[c].sum()):>5,} of {len(book_master):,}")

print(f"\ncommercially active titles: {int(book_master['has_priced_sale'].sum()):,}")
print(f"catalogue titles (all)    : {len(book_master):,}")
# ^ both counts are quoted downstream: they answer different questions.

assert book_master.index.is_unique, "isbn_key is not unique in book_master"
assert (book_master["lines"] > 0).all(), "a title with zero lines exists"

has_priced_sale      632 of 640
has_any_units        639 of 640
foc_only               4 of 640
returns_only           1 of 640
multi_desc            96 of 640

commercially active titles: 632
catalogue titles (all)    : 640


<a id="genre"></a>
## 7. Genre and sub-category

`sales_clean` carries the standardised genre vocabulary from the cleaning pass, including the client-approved sub-category family map. The dimension is built at ISBN grain and checked for internal consistency **before** the join: an ISBN carrying two different genres is a data defect that `.first()` would hide.

In [16]:
# ── 7.1 build the genre dimension at ISBN grain ─────────────────────
GENRE_COLS = [c for c in ["title", "author", "genre_std", "sub_category",
                          "subcat_family", "pub_year", "genre_source",
                          "culture_pass", "edition", "format"]
              if c in src.columns]
print(f"carrying forward: {GENRE_COLS}")

dim = src.loc[src["isbn_key"].notna(), ["isbn_key"] + GENRE_COLS].copy()

# Consistency gate: one ISBN, one genre. Runs BEFORE any collapse.
incons = {}
for c in ["genre_std", "subcat_family", "title"]:
    if c in dim:
        n = dim.groupby("isbn_key")[c].nunique(dropna=True)
        bad = n[n > 1]
        incons[c] = len(bad)
        if len(bad):
            print(f"\n!! {c}: {len(bad)} ISBNs carry >1 value, e.g. "
                  f"{[pseudonym(k, 'ISBN') for k in bad.index[:3]]}")
print(f"\ninconsistency counts: {incons}")

assert incons.get("genre_std", 0) == 0, (
    "an ISBN carries two genres: resolve upstream before joining, "
    "a .first() here would pick arbitrarily")

dim = (dim.sort_values("isbn_key")
          .groupby("isbn_key").agg(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA))
print(f"\ngenre dimension: {len(dim):,} ISBNs x {dim.shape[1]} attributes")

carrying forward: ['title', 'author', 'genre_std', 'sub_category', 'subcat_family', 'pub_year', 'genre_source', 'culture_pass', 'edition', 'format']

inconsistency counts: {'genre_std': 0, 'subcat_family': 0, 'title': 0}



genre dimension: 640 ISBNs x 10 attributes


In [17]:
# ── 7.2 join ─────────────────────────────────────────────────────────
before = len(book_master)
book_master = book_master.join(dim, how="left", rsuffix="_dim")
assert len(book_master) == before, "the join changed row count: dim is not unique"

book_master["has_genre"]  = book_master["genre_std"].notna()
book_master["has_subcat"] = (book_master["subcat_family"].notna()
                             if "subcat_family" in book_master else False)
book_master["has_title"]  = book_master["title"].notna()

print(f"titles      : {len(book_master):,}")
print(f"  w/ genre  : {int(book_master['has_genre'].sum()):,} "
      f"({book_master['has_genre'].mean()*100:.1f}%)")
print(f"  w/ subcat : {int(book_master['has_subcat'].sum()):,}")
print(f"  w/ title  : {int(book_master['has_title'].sum()):,}")

_seg = book_master.loc[book_master["has_genre"], "revenue_sale"].sum()
_all = book_master["revenue_sale"].sum()
print(f"\nrevenue coverage of the genre join: {_seg/_all*100:.2f}% of book sale revenue")
# ^ THIS is the number that matters, not the title count: missing titles
#   matter only in proportion to the money they carry.

titles      : 640
  w/ genre  : 552 (86.2%)
  w/ subcat : 552
  w/ title  : 552

revenue coverage of the genre join: 98.35% of book sale revenue


<a id="gaps"></a>
## 8. Catalogue gaps

Two data-quality lists, because they are two separate asks of the client:

1. **Catalogue gaps**: ISBNs the register sells but the masterlist has no row for. These carry no `title` either (title comes from the masterlist), so the list keys on the invoice's `Item Description`.
2. **Partial records**: titles in the masterlist with a genre but no sub-category, or the reverse.

Titles, descriptions and ISBNs are shown as aliases in this public version.

In [18]:
# ── 8.1 catalogue gaps ───────────────────────────────────────────────
gaps = book_master[~book_master["has_genre"]].copy()
gaps["Remarks"] = np.where(
    gaps["multi_desc"],
    "Sold under " + gaps["n_desc"].astype(int).astype(str) + " different Item Descriptions",
    "")
gaps.loc[gaps["foc_only"], "Remarks"] += " | FOC only, no priced sale"
gaps.loc[gaps["returns_only"], "Remarks"] += " | returns only, no forward sale"
gaps["Remarks"] = gaps["Remarks"].str.strip(" |")

GAP_COLS = ["item_desc", "n_desc", "fmt_types", "lines", "units_gross",
            "first_sale", "last_sale", "active_months", "n_customers", "Remarks"]
gaps_out = gaps.sort_values("revenue_sale", ascending=False)[GAP_COLS]

print(f"catalogue gaps: {len(gaps_out):,} ISBNs · {gaps_out['lines'].sum():,} lines "
      f"· {gaps['revenue_sale'].sum() / book_master['revenue_sale'].sum() * 100:.2f}% "
      "of book sale revenue")
print(f"  multi-description: {int(gaps['multi_desc'].sum())}")
display(mask_cols(mask_index(gaps_out.head(20), "ISBN"), {"item_desc": "Item"}))

catalogue gaps: 88 ISBNs · 1,809 lines · 1.65% of book sale revenue
  multi-description: 3


,item_desc,n_desc,fmt_types,lines,units_gross,first_sale,last_sale,active_months,n_customers,Remarks
ISBN 001,Item 012,1,Paperback,25,480.0,2014-09-30,2026-03-16,20,7,
ISBN 002,Item 013,1,Paperback,100,551.0,2011-08-17,2015-05-14,24,21,
ISBN 003,Item 014,1,Paperback,13,212.0,2018-09-19,2022-06-16,7,6,
ISBN 004,Item 015,1,Paperback,50,1048.0,2020-09-10,2025-07-10,21,6,
ISBN 005,Item 016,1,Paperback,132,534.0,2017-11-13,2026-06-16,42,25,
ISBN 006,Item 017,1,Paperback,83,464.0,2011-08-22,2017-09-22,24,26,
ISBN 007,Item 018,1,Paperback,112,327.0,2017-08-02,2021-07-31,27,22,
ISBN 008,Item 019,2,Paperback,69,487.0,2011-11-23,2023-02-28,16,20,Sold under 2 different Item Descriptions
ISBN 009,Item 020,1,Paperback,61,208.0,2019-09-25,2025-09-01,41,7,
ISBN 010,Item 021,1,Paperback,9,285.0,2019-07-29,2021-04-14,9,1,


In [19]:
# ── 8.2 partial records ──────────────────────────────────────────────
partial = book_master[book_master["has_genre"] & ~book_master["has_subcat"]].copy()
print(f"genre but no sub-category: {len(partial):,} ISBNs")
if len(partial):
    display(mask_cols(mask_index(partial[["title", "item_desc", "genre_std", "lines",
                                          "units_gross"]].head(20), "ISBN"),
                      {"title": "Title", "item_desc": "Item"}))

# Unkeyed book-scoped lines: cannot reach a title master at all
unkeyed_lines = bk[bk["isbn_key"].isna()]
if len(unkeyed_lines):
    unkeyed_out = (unkeyed_lines.groupby("Item Description")
                     .agg(lines=("Qty", "size"), units=("Qty", "sum"),
                          first=("date", "min"), last=("date", "max"))
                     .sort_values("units", ascending=False))
    print(f"\nunkeyed book-scoped descriptions: {len(unkeyed_out):,}")
    display(mask_index(unkeyed_out.head(15), "Item"))

genre but no sub-category: 0 ISBNs

unkeyed book-scoped descriptions: 44


,lines,units,first,last
Item 002,177,199.0,2019-04-05,2026-07-30
Item 032,18,150.0,2015-02-03,2024-01-02
Item 033,26,147.0,2013-11-08,2019-07-31
Item 003,103,109.0,2023-05-09,2026-07-20
Item 034,9,84.0,2021-01-21,2021-01-29
Item 009,29,43.0,2020-06-26,2021-11-10
Item 035,9,37.0,2022-11-28,2024-07-20
Item 036,15,34.0,2019-04-05,2019-04-29
Item 037,5,33.0,2015-08-31,2019-09-12
Item 038,16,31.0,2013-12-27,2018-07-25


<a id="contract"></a>
## 9. Filter contract

Every filter is composed from one book-scope predicate, so changing the book definition is one edit, not one per filter. `revenue_ts` uses a positive list of scope reasons, so a new category cannot leak in unnoticed.

In [20]:
# ── 9.1 define ───────────────────────────────────────────────────────
# MODIFY CONDITION: change BOOK_SCOPE here and nowhere else. Never inline it.
BOOK_SCOPE    = "book_scope == True"
REVENUE_SCOPE = f'({BOOK_SCOPE} or scope_reason in ["service", "institutional_bulk"])'

# Guard that can fail: catches any future scope_reason drift
_book_reasons = set(src.loc[src["book_scope"], "scope_reason"].unique())
assert _book_reasons <= {"book", "book_rescued_isbn"}, \
    f"unexpected scope_reason inside book_scope: {_book_reasons}"

HAS_CLIENT_EXCL = "client_excluded" in src.columns
CLIENT = " and client_excluded == False" if HAS_CLIENT_EXCL else ""
if not HAS_CLIENT_EXCL:
    print("[!] client_excluded absent: advance-order lines are NOT excluded")

FILTERS = {
    "composition": "index == index",
    "all":         "index == index",
    "units_observable": 'qty_sign != "null"',
    "demand":      f'{BOOK_SCOPE} and line_role in ["sale","foc","unpriced"]{CLIENT}',
    "genre":       f'{BOOK_SCOPE} and line_role in ["sale","foc","unpriced"] '
                   f'and has_genre == True{CLIENT}',
    "net_revenue": f'{BOOK_SCOPE} and qty_sign != "null"{CLIENT}',
    "returns":     f'{BOOK_SCOPE} and line_role == "return"{CLIENT}',
    "price_vol":   f'{BOOK_SCOPE} and line_role == "sale" and Qty > 0{CLIENT}',
    "revenue_ts":  f'{REVENUE_SCOPE} and line_role != "return" '
                   f'and qty_sign != "null"{CLIENT}',
}
for k, v in FILTERS.items():
    print(f"{k:<18} {v}")

composition        index == index
all                index == index
units_observable   qty_sign != "null"
demand             book_scope == True and line_role in ["sale","foc","unpriced"] and client_excluded == False
genre              book_scope == True and line_role in ["sale","foc","unpriced"] and has_genre == True and client_excluded == False
net_revenue        book_scope == True and qty_sign != "null" and client_excluded == False
returns            book_scope == True and line_role == "return" and client_excluded == False
price_vol          book_scope == True and line_role == "sale" and Qty > 0 and client_excluded == False
revenue_ts         (book_scope == True or scope_reason in ["service", "institutional_bulk"]) and line_role != "return" and qty_sign != "null" and client_excluded == False


In [21]:
# ── 9.2 attach has_genre to the line frame, then verify ─────────────
src["has_genre"] = (src["isbn_key"].map(book_master["has_genre"])
                    .astype("boolean").fillna(False).astype(bool))

POP = {}
for k, v in FILTERS.items():
    sub = src.query(v, **Q)
    POP[k] = len(sub)
    print(f"{k:<18} {len(sub):>7,} lines · {sub['Qty'].sum():>9,.0f} units "
          f"· {sub['isbn_key'].nunique():>4} ISBNs")

# Asserts capable of failing on plausible data
assert POP["demand"] + POP["returns"] == POP["net_revenue"], (
    f"GROSS + returns != NET: {POP['demand']:,} + {POP['returns']:,} "
    f"!= {POP['net_revenue']:,}")
assert POP["genre"]     <= POP["demand"],  "genre must be a subset of demand"
assert POP["price_vol"] <= POP["demand"],  "price_vol must be a subset of demand"
assert POP["all"] == len(src),             "identity filter is not identity"
assert all(v > 0 for v in POP.values()),   "a filter returned an empty frame"
assert set(src.query(FILTERS["demand"], **Q)["isbn_key"].dropna()) \
       <= set(src.query(FILTERS["revenue_ts"], **Q)["isbn_key"].dropna()), \
    "revenue_ts lost an ISBN that demand keeps: the scopes have diverged"

IDENTITY = {"all", "composition"}
subset = [k for k in POP if k not in IDENTITY]
assert all(POP[k] < len(src) for k in subset), (
    f"a subset filter dropped nothing: "
    f"{[k for k in subset if POP[k] == len(src)]}")
print(f"\n[OK] contract v{CONTRACT_VERSION} verified")

composition         94,872 lines ·   653,585 units ·  640 ISBNs
all                 94,872 lines ·   653,585 units ·  640 ISBNs
units_observable    94,510 lines ·   653,585 units ·  640 ISBNs
demand              83,346 lines ·   541,817 units ·  639 ISBNs
genre               81,336 lines ·   531,424 units ·  552 ISBNs


net_revenue         90,662 lines ·   497,985 units ·  640 ISBNs


returns              7,316 lines ·   -43,832 units ·  491 ISBNs


price_vol           74,990 lines ·   456,266 units ·  632 ISBNs
revenue_ts          83,614 lines ·   555,279 units ·  639 ISBNs



[OK] contract v1 verified


In [22]:
# ── 9.3 attrition audit: which ISBNs, and why ────────────────────────
REF = "net_revenue"        # widest book-scoped filter: the ISBN ceiling
ref = set(src.query(FILTERS[REF], **Q)["isbn_key"].dropna())

for k in ["demand", "genre", "price_vol", "revenue_ts", "returns"]:
    got  = set(src.query(FILTERS[k], **Q)["isbn_key"].dropna())
    lost = ref - got
    print(f"{k:<12} {len(got):>4} ISBNs · −{len(lost):>3}")
    if 0 < len(lost) <= 15:
        cols = [c for c in ["item_desc", "units_gross", "has_priced_sale",
                            "foc_only", "returns_only"] if c in book_master]
        display(mask_cols(mask_index(book_master.loc[sorted(lost), cols], "ISBN"),
                          {"item_desc": "Item"}))
    print()

demand        639 ISBNs · −  1


,item_desc,units_gross,has_priced_sale,foc_only,returns_only
ISBN 021,Item 044,0.0,False,False,True



genre         552 ISBNs · − 88

price_vol     632 ISBNs · −  8


,item_desc,units_gross,has_priced_sale,foc_only,returns_only
ISBN 022,Item 045,2.0,False,True,False
ISBN 023,Item 046,4.0,False,True,False
ISBN 021,Item 044,0.0,False,False,True
ISBN 024,Item 047,150.0,False,True,False
ISBN 025,Item 048,481.0,False,True,False
ISBN 026,Item 049,5.0,False,False,False
ISBN 027,Item 050,5.0,False,False,False
ISBN 028,Item 051,7.0,False,False,False


revenue_ts    639 ISBNs · −  1


,item_desc,units_gross,has_priced_sale,foc_only,returns_only
ISBN 021,Item 044,0.0,False,False,True



returns       491 ISBNs · −149



<a id="export"></a>
## 10. Export

In [23]:
# ── 10.1 scoped line frame ───────────────────────────────────────────
# No rows removed. book_scope / scope_reason / line_role / amt_sign /
# qty_sign / has_genre are ADDED.
OUT_PARQUET = PROCESSED / "sales_scoped.parquet"
assert len(src) == FROZEN_ROWS, "row count moved during the build"

src.to_parquet(OUT_PARQUET, index=False)
rt = pd.read_parquet(OUT_PARQUET)
assert rt.shape == src.shape, "parquet round-trip changed shape"
for c in ["book_scope", "scope_reason", "line_role", "amt_sign", "qty_sign", "has_genre"]:
    assert c in rt.columns, f"{c} lost in round-trip"
print(f"[OK] {OUT_PARQUET.name} · {len(rt):,} rows x {rt.shape[1]} cols")

[OK] sales_scoped.parquet · 94,872 rows x 53 cols


In [24]:
# ── 10.2 book master ─────────────────────────────────────────────────
book_master.to_csv(LOOKUP / "book_master.csv")
print(f"[OK] book_master.csv · {len(book_master):,} ISBNs")

[OK] book_master.csv · 640 ISBNs


In [25]:
# ── 10.3 filter contract ─────────────────────────────────────────────
# Nested payload: readers go through capstone.load_contract(), which checks
# the version before handing out a single filter string.
payload = {
    "contract_version": CONTRACT_VERSION,
    "built":            pd.Timestamp.now().isoformat(timespec="seconds"),
    "book_scope":       BOOK_SCOPE,
    "revenue_scope":    REVENUE_SCOPE,
    "book_types":       sorted(BOOK_TYPES),
    "frozen_rows":      int(len(src)),
    "source_parquet":   OUT_PARQUET.name,
    "control_totals":   POP,
    "filters":          FILTERS,
}
OUT_JSON = LOOKUP / "analysis_filters.json"
OUT_JSON.write_text(json.dumps(payload, indent=2))
print(f"[OK] {OUT_JSON.name} · v{CONTRACT_VERSION} · {len(FILTERS)} filters")

[OK] analysis_filters.json · v1 · 9 filters


<a id="conclusion"></a>
## 11. Conclusion

| Measure | Result |
|---|---|
| Lines in book scope | 91,258 of 94,872 (96.19%), including 1 line rescued on ISBN evidence |
| Titles (ISBNs) in the master | 640, of which 632 are commercially active |
| Titles with a genre record | 552 (86.2%), carrying 98.35% of book sale revenue |
| Catalogue gaps | 88 ISBNs, 1.65% of book sale revenue |
| Filters in the contract | 9, each verified against its control total |

* **Every excluded line has a stated reason.** Defining a book line positively from `Type`, with service and institutional carve-outs taking precedence, sets aside 3,614 lines (services, shipping, vouchers, bulk orders) and records why in `scope_reason`.
* **Money and units are classified separately.** The four-state sign taxonomy yields six mutually exclusive line roles, so returns (6.19% of sale revenue) and discounts can never be confused.
* **One definition, many readers.** Notebooks 2 to 4 read these populations through `capstone.py`, so a change to the book definition is one edit here, not one per notebook.